# Controlled Network Centrality & Degree Distribution Study: BA vs. ER

This interactive notebook demonstrates the step-by-step exploration of:
1. **Fair network generation** with matched size ($N=100$) and edge count ($M=196$).
2. **Comprehensive centrality computation** (9 measures + structural descriptors).
3. **Degree distribution analysis**, Poisson comparison, and power-law MLE fitting.
4. **Comparative topological analysis**, Gini inequality, and percolation robustness.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.generate import generate_networks
from src.centrality import compute_centralities, get_top_k_nodes, gini_coefficient
from src.degree_dist import compute_degree_distribution, theoretical_poisson_pmf, fit_power_law

## 1. Network Generation Under Controlled Constraints
We generate Barabási–Albert ($m=2$) and Erdős–Rényi $G(n, M)$ with identical edge counts ($M=196$).

In [ ]:
G_ba, G_er, diag = generate_networks(n=100, m_ba=2, seed=42)
print(f"BA Nodes: {G_ba.number_of_nodes()}, Edges: {G_ba.number_of_edges()}, Connected: {diag['ba']['is_connected']}")
print(f"ER Nodes: {G_er.number_of_nodes()}, Edges: {G_er.number_of_edges()}, Connected: {diag['er']['is_connected']}, Components: {diag['er']['component_sizes']}")

## 2. Centrality Computations and Spectral Bounds
Calculate all 9 centrality measures, spectral radius $\lambda_{\max}$, and Gini inequality.

In [ ]:
df_ba, meta_ba = compute_centralities(G_ba)
df_er, meta_er = compute_centralities(G_er)

print(f"Spectral Radius lambda_max: BA = {meta_ba['lambda_max']:.3f}, ER = {meta_er['lambda_max']:.3f}")
print(f"Katz alpha threshold (1/lambda_max): BA = {meta_ba['alpha_katz_upper_bound']:.4f}, ER = {meta_er['alpha_katz_upper_bound']:.4f}")
print(f"Chosen Katz alpha: {meta_ba['alpha_katz_chosen']:.4f}")

gini_comp = pd.DataFrame({
    'BA Gini': meta_ba['gini_coefficients'],
    'ER Gini': meta_er['gini_coefficients']
})
gini_comp['Ratio (BA/ER)'] = (gini_comp['BA Gini'] / np.maximum(1e-5, gini_comp['ER Gini'])).round(2)
gini_comp

## 3. Top-10 Ranked Nodes & First-Mover Advantage

In [ ]:
top_ba = get_top_k_nodes(df_ba, k=10)
top_ba[['degree_node', 'degree_val', 'betweenness_node', 'betweenness_val', 'eigenvector_node', 'eigenvector_val']]

## 4. Power-Law Fitting on Barabási–Albert Network

In [ ]:
deg_ba, k_ba, p_ba, ccdf_ba = compute_degree_distribution(G_ba)
fit_res = fit_power_law(deg_ba)
print(f"Estimated gamma: {fit_res['gamma']:.3f} (Asymptotic theory = 3.0)")
print(f"Estimated x_min: {fit_res['xmin']}")
print(f"Log-likelihood ratio vs exponential R: {fit_res['loglikelihood_ratio']:.3f} (p-value: {fit_res['p_value']:.4f})")